In [ ]:
# 导入必要的库
import os
import time
import random
import json
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torchinfo import summary
import matplotlib.pyplot as plt
import matplotlib.patches as mpts
from sklearn.decomposition import PCA
from sklearn.metrics import roc_curve, auc
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, recall_score, cohen_kappa_score, accuracy_score
from sklearn.metrics import precision_score, precision_recall_curve, auc, recall_score, f1_score, accuracy_score, confusion_matrix
from sklearn.preprocessing import minmax_scale
import pandas as pd
from scipy.io import loadmat
from tqdm import tqdm
from IPython import display
import h5py
import copy
import sys
import glob
import seaborn as sns
from datetime import datetime
import optuna
from optuna.visualization import plot_param_importances, plot_optimization_history
%matplotlib inline

In [ ]:
##超参数和实验设置
RANDOM_SEED = 666
MODEL_NAME = 'BrainVoxel_102Class_MLP'  # 修改为MLP模型名称
DATASET = 'BrainVoxel'  # 数据集名称
APPLY_PCA = False   # 是否应用PCA降维
N_PCA = 0          # PCA保留的主成分数量，0表示不进行PCA
NORM = True        # 数据标准化

# 训练参数
EPOCH = 30         # 总训练轮数
VAL_EPOCH = 3      # 验证频率
LR = 1e-5          # 学习率 - 将由贝叶斯优化确定
WEIGHT_DECAY = 1e-5  # 权重衰减系数 - 将由贝叶斯优化确定
BATCH_SIZE = 128    # 批处理大小 - 将由贝叶斯优化确定

# 学习率调度参数
USE_LR_SCHEDULER = True           # 是否使用学习率调度
LR_SCHEDULER_TYPE = 'cosine'   # 调度器类型: 'multistep', 'cosine', 'plateau'
LR_MILESTONES = [10, 20]      # 多步调度的里程碑轮次
LR_GAMMA = 0.5                 # 学习率降低的倍数因子

# 计算设备选择
DEVICE = 0         # -1表示CPU，0表示第一块GPU

# 数据处理选项 - 移除平衡处理
BALANCE_TRAIN = False  # 不进行类别平衡处理
TARGET_SAMPLES = 0     # 不再使用

# 数据参数
FEATURE_DIM = 341  # 原始特征维度
NUM_CLASS = 102    # 类别数量
DROPOUT_RATE = 0.5 # Dropout率 - 将由贝叶斯优化确定

# 新增MLP架构参数
HIDDEN_UNITS = [4096, 4096, 4096, 4096]  # 四层4096宽度的MLP
ACTIVATION = 'relu'  # 激活函数

# 结果保存路径
SAVE_PATH = f"./Results/{MODEL_NAME}/{DATASET}"
if not os.path.isdir(SAVE_PATH):
    os.makedirs(SAVE_PATH)

In [ ]:
## 设置随机数种子，确保实验结果可复现

# 为Python的random模块设置随机种子
random.seed(RANDOM_SEED)

# 为PyTorch的CPU操作设置随机种子
torch.manual_seed(RANDOM_SEED)

# 为当前GPU设置随机种子
torch.cuda.manual_seed(RANDOM_SEED)

# 为所有可用GPU设置相同的随机种子
torch.cuda.manual_seed_all(RANDOM_SEED)

# 为NumPy库设置随机种子
np.random.seed(RANDOM_SEED)

# 禁用CuDNN的非确定性算法
torch.backends.cudnn.deterministic = True

# 禁用CuDNN的自动优化选择
torch.backends.cudnn.benchmark = False



In [ ]:
# 灵活的采样器和数据管理类
class BrainVoxelSampler:
    """脑体素数据采样器，提供多种采样策略"""
    
    def __init__(self, data_dir):
        """
        初始化采样器
        
        参数:
            data_dir: 数据集目录
        """
        self.data_dir = data_dir
        self.label_info = self._load_label_index()
        self.valid_labels = [label for label, info in self.label_info.items() if info['count'] > 0]
    
    def _load_label_index(self):
        """加载标签索引文件"""
        index_file = os.path.join(self.data_dir, "label_index.txt")
        label_info = {}
        
        with open(index_file, 'r') as f:
            # 跳过表头
            next(f)
            for line in f:
                parts = line.strip().split(',')
                if len(parts) >= 3:
                    label_id = int(parts[0])
                    voxel_count = int(parts[1])
                    filename = parts[2] if parts[2] else None
                    label_info[label_id] = {'count': voxel_count, 'filename': filename}
        
        return label_info
    
    def get_file_path(self, label_id):
        """获取指定标签的文件路径"""
        if label_id not in self.label_info:
            return None
        
        filename = self.label_info[label_id]['filename']
        if not filename:
            return None
            
        return os.path.join(self.data_dir, filename)

In [ ]:
# 脑体素数据载入工具函数
def load_brain_voxel_data():
    """
    载入脑体素数据、标签和训练/验证/测试集
    
    返回:
        data: 高维特征数据
        train_gt: 训练集标签
        val_gt: 验证集标签
        all_data_dict: 包含所有数据集信息的字典
    """
    # 路径配置 - 根据您的实际路径进行调整
    output_path = '/home/jovyan/gpu_space/workspace_jiayi/KAN training/brain_voxel_data/output'
    train_label_dir = os.path.join(output_path, 'train_set_by_label')
    val_label_dir = os.path.join(output_path, 'val_set_by_label')
    
    # 读取标签索引文件
    def load_label_index(index_file):
        label_info = {}
        with open(index_file, 'r') as f:
            # 跳过表头
            next(f)
            for line in f:
                parts = line.strip().split(',')
                if len(parts) >= 3:
                    label_id = int(parts[0])
                    voxel_count = int(parts[1])
                    filename = parts[2] if parts[2] else None
                    label_info[label_id] = {'count': voxel_count, 'filename': filename}
        return label_info
    
    train_index_file = os.path.join(train_label_dir, "label_index.txt")
    val_index_file = os.path.join(val_label_dir, "val_label_index.txt")
    
    if not os.path.exists(train_index_file):
        raise FileNotFoundError(f"训练标签索引文件不存在: {train_index_file}")
    if not os.path.exists(val_index_file):
        raise FileNotFoundError(f"验证标签索引文件不存在: {val_index_file}")
    
    train_label_info = load_label_index(train_index_file)
    val_label_info = load_label_index(val_index_file)
    
    # 获取有效标签（有体素数据的标签）
    valid_labels = [label_id for label_id, info in train_label_info.items() 
                   if info['count'] > 0]
    
    print(f"数据加载完成: 找到 {len(valid_labels)} 个有效标签")
    
    return {
        'train_label_info': train_label_info,
        'val_label_info': val_label_info,
        'valid_labels': valid_labels,
        'train_label_dir': train_label_dir,
        'val_label_dir': val_label_dir
    }

# 加载数据集信息
all_data_dict = load_brain_voxel_data()

In [ ]:
def apply_pca(X, num_components=15, norm=True, pca_model=None):
    """
    对数据进行PCA降维和标准化处理
    
    参数:
        X (ndarray): 需要降维的数据
        num_components (int): 保留的主成分数量，0表示不进行PCA
        norm (bool): 是否进行标准化处理
        pca_model: 预先训练好的PCA模型，None表示需要重新拟合
    
    返回:
        new_X: 处理后的数据
        num_components: 最终的特征维度
        pca_model: 使用或训练的PCA模型
    """
    if num_components == 0:
        # 不进行PCA，但可能进行标准化
        if norm:
            # 对每个特征进行标准化
            mean = np.mean(X, axis=0)
            std = np.std(X, axis=0)
            # 避免除以0
            std[std == 0] = 1
            new_X = (X - mean) / std
        else:
            new_X = X.copy()
        return new_X, X.shape[1], None
    else:
        # 进行PCA降维
        if pca_model is None:
            # 如果没有提供PCA模型，则训练一个新的
            pca_model = PCA(n_components=num_components)
            new_X = pca_model.fit_transform(X)
        else:
            # 使用提供的PCA模型转换数据
            new_X = pca_model.transform(X)
        
        # 可选的标准化
        if norm:
            # 对PCA后的特征进行归一化
            new_X = (new_X - np.min(new_X, axis=0)) / (np.max(new_X, axis=0) - np.min(new_X, axis=0) + 1e-10)
        
        return new_X, new_X.shape[1], pca_model

def analyze_pca_variance(X, max_components=None, plot=True, save_path=None):
    """
    分析PCA的方差解释率，找到合适的降维维度
    
    参数:
        X (ndarray): 输入数据
        max_components (int): 最大考虑的主成分数，None表示使用特征维度
        plot (bool): 是否绘制解释方差曲线
        save_path (str): 保存图像的路径，None表示不保存
        
    返回:
        optimal_n_components: 建议的主成分数量
    """
    # 确定最大主成分数
    if max_components is None:
        max_components = min(X.shape[0], X.shape[1])
    else:
        max_components = min(max_components, X.shape[0], X.shape[1])
    
    # 计算所有可能的主成分
    pca = PCA(n_components=max_components)
    pca.fit(X)
    
    # 计算累积解释方差
    explained_variance_ratio = pca.explained_variance_ratio_
    cumulative_variance_ratio = np.cumsum(explained_variance_ratio)
    
    # 寻找方差解释率达到95%的拐点
    threshold = 0.95
    optimal_n_components = np.argmax(cumulative_variance_ratio >= threshold) + 1
    
    # 寻找拐点（斜率变化最大的点）
    gradient = np.gradient(explained_variance_ratio)
    gradient_of_gradient = np.gradient(gradient)
    elbow_index = np.argmax(np.abs(gradient_of_gradient))
    elbow_n_components = elbow_index + 1
    

    if plot:
        plt.figure(figsize=(12, 6))
    
        # Plot Explained Variance Ratio
        plt.subplot(1, 2, 1)
        plt.plot(range(1, len(explained_variance_ratio) + 1), 
                 explained_variance_ratio, 'bo-', markersize=4)
        plt.axvline(x=elbow_n_components, color='r', linestyle='--', 
                    label=f'Elbow Point: {elbow_n_components} Components')
        plt.xlabel('Number of Principal Components')
        plt.ylabel('Explained Variance Ratio')
        plt.title('Explained Variance Ratio per Principal Component')
        plt.grid(True)
        plt.legend()
    
        # Plot Cumulative Explained Variance
        plt.subplot(1, 2, 2)
        plt.plot(range(1, len(cumulative_variance_ratio) + 1), 
                 cumulative_variance_ratio, 'ro-', markersize=4)
        plt.axhline(y=threshold, color='g', linestyle='--', 
                    label=f'{threshold*100}% Variance')
        plt.axvline(x=optimal_n_components, color='b', linestyle='--', 
                    label=f'Threshold Components: {optimal_n_components}')
        plt.xlabel('Number of Principal Components')
        plt.ylabel('Cumulative Explained Variance Ratio')
        plt.title('Cumulative Explained Variance Ratio')
        plt.grid(True)
        plt.legend()
    
        plt.tight_layout()
    
        if save_path:
            plt.savefig(save_path)
        plt.show()

    
    print(f"方差拐点对应的主成分数量: {elbow_n_components}")
    print(f"达到{threshold*100}%方差解释率需要的主成分数量: {optimal_n_components}")
    print(f"前{optimal_n_components}个主成分解释了总方差的{cumulative_variance_ratio[optimal_n_components-1]*100:.2f}%")
    
    # 修改为使用95%阈值点
    suggested_components = optimal_n_components  # 使用保留95%信息的维度
    return suggested_components, explained_variance_ratio, cumulative_variance_ratio


In [ ]:
class BrainVoxelDataset(Dataset):
    """
    脑体素数据集类，多分类版本
    """
    def __init__(self, data, labels):
        """
        初始化数据集
        
        参数:
            data: 特征数据，形状为(n_samples, feature_dim)
            labels: 标签数据，形状为(n_samples,)
        """
        super(BrainVoxelDataset, self).__init__()
        self.data = data
        self.labels = labels
        
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        x = self.data[idx]
        x = torch.FloatTensor(x)
        
        y = self.labels[idx]
        # 对于多分类问题，标签从0开始到101
        if y == 0:  # 背景像素
            y = -1  # 设置为忽略索引
        else:
            y = y - 1  # 将1-102转为0-101
            
        y = torch.LongTensor([int(y)])[0]
        return x, y

In [ ]:
def load_multiclass_data(data_dirs, apply_pca_flag=True, n_components=24, norm=True):
    """
    载入所有类别的数据用于多分类训练 - 不进行类别平衡处理
    
    参数:
        data_dirs: 包含训练、测试和验证数据目录的字典
        apply_pca_flag: 是否应用PCA降维
        n_components: PCA保留的主成分数量，0表示自动选择
        norm: 是否进行标准化处理
        
    返回:
        dataset_dict: 包含训练、测试和验证集数据的字典
    """
    # 创建数据采样器
    train_sampler = BrainVoxelSampler(data_dirs['train_dir'])
    test_sampler = BrainVoxelSampler(data_dirs['test_dir'])
    val_sampler = BrainVoxelSampler(data_dirs['val_dir'])
    
    # 收集所有有效标签
    valid_labels = sorted(list(set(
        train_sampler.valid_labels + 
        test_sampler.valid_labels + 
        val_sampler.valid_labels
    )))
    
    print(f"找到 {len(valid_labels)} 个有效标签")
    
    # 收集所有训练集样本和标签
    train_samples = []
    train_labels = []
    
    for label_id in tqdm(valid_labels, desc="加载训练集数据"):
        file_path = train_sampler.get_file_path(label_id)
        if file_path:
            samples = np.load(file_path)
            labels = np.ones(len(samples)) * label_id
            train_samples.append(samples)
            train_labels.append(labels)
    
    # 收集所有测试集样本和标签
    test_samples = []
    test_labels = []
    
    for label_id in tqdm(valid_labels, desc="加载测试集数据"):
        file_path = test_sampler.get_file_path(label_id)
        if file_path:
            samples = np.load(file_path)
            labels = np.ones(len(samples)) * label_id
            test_samples.append(samples)
            test_labels.append(labels)
    
    # 收集所有验证集样本和标签
    val_samples = []
    val_labels = []
    
    for label_id in tqdm(valid_labels, desc="加载验证集数据"):
        file_path = val_sampler.get_file_path(label_id)
        if file_path:
            samples = np.load(file_path)
            labels = np.ones(len(samples)) * label_id
            val_samples.append(samples)
            val_labels.append(labels)
    
    # 合并各自的数据
    train_samples = np.vstack(train_samples) if train_samples else np.array([])
    train_labels = np.concatenate(train_labels) if train_labels else np.array([])
    test_samples = np.vstack(test_samples) if test_samples else np.array([])
    test_labels = np.concatenate(test_labels) if test_labels else np.array([])
    val_samples = np.vstack(val_samples) if val_samples else np.array([])
    val_labels = np.concatenate(val_labels) if val_labels else np.array([])
    
    # 打印数据集统计信息
    print(f"\n{'='*60}\n多分类数据集统计信息\n{'='*60}")
    print(f"训练集: {len(train_labels)} 个样本")
    print(f"测试集: {len(test_labels)} 个样本")
    print(f"验证集: {len(val_labels)} 个样本")
    
    # 应用PCA（如果需要）
    if apply_pca_flag:
        # 合并所有数据进行PCA拟合
        all_samples = np.vstack([train_samples, test_samples, val_samples])
        
        if n_components == 0:
            # 自动选择主成分数量
            n_components, _, _ = analyze_pca_variance(all_samples, plot=True)
        
        # 创建并拟合PCA模型
        pca_model = PCA(n_components=n_components)
        pca_model.fit(all_samples)
        
        # 应用PCA变换
        train_samples = pca_model.transform(train_samples)
        test_samples = pca_model.transform(test_samples)
        val_samples = pca_model.transform(val_samples)
        
        # 应用标准化（如果需要）
        if norm:
            # 基于所有样本计算归一化参数
            all_transformed = np.vstack([train_samples, test_samples, val_samples])
            mins = np.min(all_transformed, axis=0)
            maxs = np.max(all_transformed, axis=0)
            ranges = maxs - mins + 1e-10  # 避免除零
            
            # 应用归一化
            train_samples = (train_samples - mins) / ranges
            test_samples = (test_samples - mins) / ranges
            val_samples = (val_samples - mins) / ranges
        
        feature_dim = train_samples.shape[1]
        print(f"\n特征维度: {feature_dim} (PCA降维后)")
    else:
        feature_dim = train_samples.shape[1]
        print(f"\n特征维度: {feature_dim} (原始特征)")
        pca_model = None
    
    # 类别分布统计
    print("\n类别分布:")
    class_counts = {}
    for label_id in valid_labels:
        train_count = np.sum(train_labels == label_id)
        test_count = np.sum(test_labels == label_id)
        val_count = np.sum(val_labels == label_id)
        total_count = train_count + test_count + val_count
        print(f"标签 {label_id}: 训练集 {train_count}, 测试集 {test_count}, 验证集 {val_count}, 总计 {total_count}")
        class_counts[label_id] = {
            'train': train_count,
            'test': test_count,
            'val': val_count,
            'total': total_count
        }
    
    # 显示某些关键的类别统计
    print("\n类别统计摘要:")
    min_label = min(class_counts.items(), key=lambda x: x[1]['total'])[0]
    max_label = max(class_counts.items(), key=lambda x: x[1]['total'])[0]
    print(f"样本最少的类别: 标签 {min_label}, 共 {class_counts[min_label]['total']} 个样本")
    print(f"样本最多的类别: 标签 {max_label}, 共 {class_counts[max_label]['total']} 个样本")
    print(f"类别不平衡比例: {class_counts[max_label]['total'] / class_counts[min_label]['total']:.2f} : 1")
    
    # 统计小样本类别
    small_classes = [k for k, v in class_counts.items() if v['total'] < 10]
    if small_classes:
        print(f"\n样本数少于10的类别: {len(small_classes)} 个")
        for label in small_classes:
            print(f"标签 {label}: {class_counts[label]['total']} 个样本")
    
    return {
        'train_samples': train_samples,
        'train_labels': train_labels,
        'test_samples': test_samples,
        'test_labels': test_labels,
        'val_samples': val_samples,
        'val_labels': val_labels,
        'feature_dim': feature_dim,
        'pca_model': pca_model,
        'valid_labels': valid_labels,
        'class_counts': class_counts
    }

In [ ]:
class BrainVoxelMLP(nn.Module):
    """
    用于脑体素分类的多层感知器模型，多分类版本
    """
    def __init__(self, input_dim, hidden_dims, num_classes, dropout_rate=0.5, activation='relu'):
        """
        初始化模型
        
        参数:
            input_dim: 输入特征维度
            hidden_dims: 隐藏层维度列表，例如[4096, 4096, 4096, 4096]
            num_classes: 类别数量 (102)
            dropout_rate: Dropout比率
            activation: 激活函数，支持'relu'和'gelu'
        """
        super(BrainVoxelMLP, self).__init__()
        
        self.layers = nn.ModuleList()
        
        # 添加输入层到第一个隐藏层
        if isinstance(hidden_dims, int):
            hidden_dims = [hidden_dims]
        
        # 输入层到第一个隐藏层
        self.layers.append(nn.Linear(input_dim, hidden_dims[0]))
        
        # 选择激活函数
        if activation == 'relu':
            act_fn = nn.ReLU()
        elif activation == 'gelu':
            act_fn = nn.GELU()
        elif activation == 'swish':
            act_fn = nn.SiLU()  # PyTorch中的SiLU就是Swish激活函数
        else:
            act_fn = nn.ReLU()  # 默认使用ReLU
            
        self.layers.append(act_fn)
        self.layers.append(nn.Dropout(dropout_rate))
        
        # 添加中间隐藏层
        for i in range(len(hidden_dims) - 1):
            self.layers.append(nn.Linear(hidden_dims[i], hidden_dims[i+1]))
            self.layers.append(act_fn)
            self.layers.append(nn.Dropout(dropout_rate))
        
        # 最后的分类层
        self.layers.append(nn.Linear(hidden_dims[-1], num_classes))
    
    def forward(self, x):
        """
        前向传播
        
        参数:
            x: 输入特征，形状为(batch_size, input_dim)
            
        返回:
            output: 模型输出，形状为(batch_size, num_classes)
        """
        for layer in self.layers:
            x = layer(x)
        return x

In [ ]:
def calculate_class_weights(train_labels, num_classes=NUM_CLASS):
    """
    计算类别权重，解决不平衡问题
    """
    # 统计每个类别的样本数
    class_counts = {}
    for i in range(1, num_classes+1):  # 原始标签从1开始到102
        class_counts[i] = np.sum(train_labels == i)
    
    # 创建权重数组
    weights = np.zeros(num_classes)
    
    # 计算权重（反比于频率）
    for i in range(1, num_classes+1):
        count = max(class_counts.get(i, 0), 1)  # 避免除零
        weights[i-1] = 1.0 / count  # 权重索引从0开始
    
    # 归一化权重
    weights = weights / weights.sum() * len(weights)
    
    return torch.FloatTensor(weights)

def train_brain_voxel_mlp_multiclass(model, train_loader, val_loader, criterion, optimizer, device, 
                          num_epochs=100, val_epoch=1, save_path="./Results",
                          lr_scheduler=None):
    """
    训练脑体素MLP多分类模型，并输出训练集和验证集的性能指标
    
    参数:
        model: MLP模型
        train_loader: 训练数据加载器
        val_loader: 验证数据加载器
        criterion: 损失函数
        optimizer: 优化器
        device: 计算设备
        num_epochs: 训练轮数
        val_epoch: 验证频率
        save_path: 模型保存路径
        lr_scheduler: 学习率调度器
    
    返回:
        训练结果统计信息
    """
    # 初始化统计变量
    loss_list = []
    acc_list = []
    f1_macro_list = []  # 训练集F1宏平均记录
    
    val_acc_list = []
    val_epoch_list = []
    val_f1_macro_list = []
    val_kappa_list = []
    val_balanced_acc_list = []
    lr_list = []  # 学习率列表
    e = 0  # 初始化epoch计数器
    
    # 保存起始时间
    train_st = time.time()
    
    # 计算批次数量和样本数量
    batch_num = len(train_loader)
    train_num = len(train_loader.dataset)
    val_num = len(val_loader.dataset)
    
    try:
        # 训练循环
        for e in tqdm(range(num_epochs), desc="Training Progress:"):
            # 获取当前学习率
            current_lr = optimizer.param_groups[0]['lr']
            lr_list.append(current_lr)
            
            # 设置模型为训练模式
            model.train()
            avg_loss = 0.0
            train_acc = 0
            valid_count = 0
            
            # 收集训练集的预测结果
            train_all_preds = []
            train_all_targets = []
            
            # 批次循环
            for batch_idx, (data, target) in tqdm(enumerate(train_loader), total=batch_num):
                # 将数据移动到指定设备
                data, target = data.to(device), target.to(device)
                
                # 前向传播
                optimizer.zero_grad()
                out = model(data)
                loss = criterion(out, target)
                
                # 反向传播
                loss.backward()
                optimizer.step()
                
                # 累计损失和准确率
                avg_loss += loss.item()
                _, pred = torch.max(out, dim=1)
                valid_mask = target != -1  # 忽略背景(-1)的准确率计算
                train_acc += (pred[valid_mask] == target[valid_mask]).sum().item()
                valid_count += valid_mask.sum().item()
                
                # 收集预测和目标用于计算F1等指标
                train_all_preds.extend(pred[valid_mask].cpu().numpy())
                train_all_targets.extend(target[valid_mask].cpu().numpy())
            
            # 计算本轮平均损失和准确率
            loss_list.append(avg_loss / batch_num)
            acc_list.append(train_acc / valid_count if valid_count > 0 else 0)
            
            # 计算训练集的F1分数
            train_all_preds = np.array(train_all_preds)
            train_all_targets = np.array(train_all_targets)
            train_f1_macro = f1_score(train_all_targets, train_all_preds, average='macro')
            f1_macro_list.append(train_f1_macro)
            
            print(f"Epoch {e}/{num_epochs} Loss:{loss_list[-1]:.4f} Train Acc:{acc_list[-1]:.4f} Train F1:{train_f1_macro:.4f} LR:{current_lr:.6f}")
            
            # 验证阶段
            if (e+1) % val_epoch == 0 or (e+1) == num_epochs:
                val_acc = 0
                valid_count = 0
                model.eval()
                
                # 收集验证数据的预测结果
                all_preds = []
                all_targets = []
                
                with torch.no_grad():
                    for batch_idx, (data, target) in tqdm(enumerate(val_loader), total=len(val_loader)):
                        data, target = data.to(device), target.to(device)
                        out = model(data)
                        _, pred = torch.max(out, dim=1)
                        
                        # 收集有效预测（非背景）
                        valid_mask = target != -1
                        all_preds.extend(pred[valid_mask].cpu().numpy())
                        all_targets.extend(target[valid_mask].cpu().numpy())
                        val_acc += (pred[valid_mask] == target[valid_mask]).sum().item()
                        valid_count += valid_mask.sum().item()
                
                # 计算全面的评估指标
                all_preds = np.array(all_preds)
                all_targets = np.array(all_targets)
                val_accuracy = val_acc / valid_count if valid_count > 0 else 0
                val_f1_macro = f1_score(all_targets, all_preds, average='macro')
                val_kappa = cohen_kappa_score(all_targets, all_preds)
                val_balanced_acc = balanced_accuracy_score(all_targets, all_preds)
                
                # 保存验证结果
                val_acc_list.append(val_accuracy)
                val_epoch_list.append(e)
                val_f1_macro_list.append(val_f1_macro)
                val_kappa_list.append(val_kappa)
                val_balanced_acc_list.append(val_balanced_acc)
                
                # 显示对比训练集和验证集的评估指标
                train_val_diff = train_f1_macro - val_f1_macro  # 训练集和验证集F1的差异（用于评估过拟合）
                
                print(f"Epoch {e}/{num_epochs}")
                print(f"  Train: Acc:{acc_list[-1]:.4f}  F1:{train_f1_macro:.4f}")
                print(f"  Val:   Acc:{val_accuracy:.4f}  F1:{val_f1_macro:.4f}  Kappa:{val_kappa:.4f}  Balanced Acc:{val_balanced_acc:.4f}")
                print(f"  Diff:  F1:{train_val_diff:.4f} (Training-Validation)")

                # 保存当前模型
                save_name = os.path.join(save_path, f"epoch_{e}_acc_{val_accuracy:.4f}_f1_{val_f1_macro:.4f}.pth")

                # 保存模型和训练信息
                save_dict = {
                    'state_dict': model.state_dict(), 
                    'epoch': e+1, 
                    'optimizer': optimizer.state_dict(),
                    'loss_list': loss_list, 
                    'acc_list': acc_list,
                    'f1_macro_list': f1_macro_list,
                    'val_acc_list': val_acc_list, 
                    'val_epoch_list': val_epoch_list,
                    'val_f1_macro_list': val_f1_macro_list,
                    'val_kappa_list': val_kappa_list,
                    'val_balanced_acc_list': val_balanced_acc_list,
                    'lr_list': lr_list
                }
                torch.save(save_dict, save_name)
                
                # 更新ReduceLROnPlateau类型的学习率调度器
                if lr_scheduler is not None and isinstance(lr_scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
                    lr_scheduler.step(val_f1_macro)  # 使用验证集F1宏平均指导学习率调度
            
            # 更新其他类型的学习率调度器
            if lr_scheduler is not None and not isinstance(lr_scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
                lr_scheduler.step()
                
    except Exception as exc:
        print(exc)
        import traceback
        traceback.print_exc()
        
    finally:
        print(f'Training stopped at epoch {e}')
    
    # 计算总训练时间
    train_time = time.time() - train_st
    print(f"Training time: {train_time:.2f} seconds")
    
    # 返回训练结果
    return {
        'loss_list': loss_list,
        'acc_list': acc_list,
        'f1_macro_list': f1_macro_list,
        'val_acc_list': val_acc_list,
        'val_epoch_list': val_epoch_list,
        'val_f1_macro_list': val_f1_macro_list,
        'val_kappa_list': val_kappa_list,
        'val_balanced_acc_list': val_balanced_acc_list,
        'train_time': train_time,
        'lr_list': lr_list
    }

In [ ]:
def evaluate_model_multiclass(model, data_loader, device, class_names=None):
    """
    评估多分类模型性能
    
    参数:
        model: 训练好的模型
        data_loader: 数据加载器
        device: 计算设备
        class_names: 类别名称列表
    
    返回:
        评估结果
    """
    model.eval()
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for data, target in tqdm(data_loader, desc="评估中"):
            data, target = data.to(device), target.to(device)
            output = model(data)
            _, preds = torch.max(output, 1)
            
            # 只评估非背景像素
            valid_mask = target != -1
            all_preds.extend(preds[valid_mask].cpu().numpy())
            all_targets.extend(target[valid_mask].cpu().numpy())
    
    # 转换为numpy数组
    all_preds = np.array(all_preds)
    all_targets = np.array(all_targets)
    
    # 计算各种评估指标
    accuracy = accuracy_score(all_targets, all_preds)
    balanced_acc = balanced_accuracy_score(all_targets, all_preds)
    f1_macro = f1_score(all_targets, all_preds, average='macro')
    f1_weighted = f1_score(all_targets, all_preds, average='weighted')
    kappa = cohen_kappa_score(all_targets, all_preds)
    
    # 计算每个类的精确率、召回率和F1分数
    class_precision = precision_score(all_targets, all_preds, average=None, zero_division=0)
    class_recall = recall_score(all_targets, all_preds, average=None, zero_division=0)
    class_f1 = f1_score(all_targets, all_preds, average=None, zero_division=0)
    
    # 生成分类报告
    target_names = class_names if class_names else [f"Class {i}" for i in range(NUM_CLASS)]
    report = classification_report(all_targets, all_preds, target_names=target_names)
    
    # 生成混淆矩阵
    conf_matrix = confusion_matrix(all_targets, all_preds)
    
    # 返回评估结果
    return {
        'accuracy': accuracy,
        'balanced_accuracy': balanced_acc,
        'f1_macro': f1_macro,
        'f1_weighted': f1_weighted,
        'kappa': kappa,
        'class_precision': class_precision,
        'class_recall': class_recall,
        'class_f1': class_f1,
        'report': report,
        'confusion_matrix': conf_matrix,
        'predictions': all_preds,
        'targets': all_targets
    }

def get_best_model(metrics_list, epoch_list, save_path, metric='f1', del_others=False):
    """
    通过指定评估指标找到最佳模型
    
    参数:
        metrics_list: 指标列表（如准确率、F1或AUC-PR）
        epoch_list: 对应的epoch列表
        save_path: 模型保存路径
        metric: 要使用的指标，默认为'f1'，可选'acc'
        del_others: 是否删除其他模型
    
    返回:
        best_model_path: 最佳模型路径
    """
    metrics_list = np.array(metrics_list)
    epoch_list = np.array(epoch_list)
    best_index = np.argwhere(metrics_list == np.max(metrics_list))[-1].item()
    best_epoch = epoch_list[best_index]
    best_metric = metrics_list[best_index]
    
    # 根据使用的指标查找对应模型文件
    if metric == 'f1':
        pattern = f"epoch_{best_epoch}_*_f1_{best_metric:.4f}*.pth"
    else:  # 默认使用acc
        pattern = f"epoch_{best_epoch}_acc_{best_metric:.4f}*.pth"
    
    matching_files = glob.glob(os.path.join(save_path, pattern))
    if not matching_files:
        # 备用搜索方式
        all_model_files = glob.glob(os.path.join(save_path, "*.pth"))
        for file in all_model_files:
            if f"epoch_{best_epoch}_" in file:
                matching_files.append(file)
    
    if not matching_files:
        raise FileNotFoundError(f"找不到对应的模型文件: {pattern}")
    
    best_model_path = matching_files[0]
    print(f"最佳模型 ({metric}={best_metric:.4f}): {os.path.basename(best_model_path)}")
    
    # 删除其他模型
    if del_others:
        for f in os.listdir(save_path):
            if f.endswith('.pth') and os.path.join(save_path, f) != best_model_path:
                os.remove(os.path.join(save_path, f))
    
    return best_model_path

In [ ]:
# 载入和准备所有类别的数据
def load_all_classes_data(data_dirs, apply_pca_flag=True, n_components=24, norm=True):
    """
    载入所有类别的数据用于多分类训练
    
    参数:
        data_dirs: 包含训练、测试和验证数据目录的字典
        apply_pca_flag: 是否应用PCA降维
        n_components: PCA保留的主成分数量，0表示自动选择
        norm: 是否进行标准化处理
        
    返回:
        dataset_dict: 包含训练、测试和验证集数据的字典
    """
    # 创建数据采样器
    train_sampler = BrainVoxelSampler(data_dirs['train_dir'])
    test_sampler = BrainVoxelSampler(data_dirs['test_dir'])
    val_sampler = BrainVoxelSampler(data_dirs['val_dir'])
    
    # 收集所有有效标签
    valid_labels = sorted(list(set(
        train_sampler.valid_labels + 
        test_sampler.valid_labels + 
        val_sampler.valid_labels
    )))
    
    print(f"找到 {len(valid_labels)} 个有效标签")
    
    # 收集所有训练集样本和标签
    train_samples = []
    train_labels = []
    
    for label_id in tqdm(valid_labels, desc="加载训练集数据"):
        file_path = train_sampler.get_file_path(label_id)
        if file_path:
            samples = np.load(file_path)
            labels = np.ones(len(samples)) * label_id  # 使用实际的标签ID
            train_samples.append(samples)
            train_labels.append(labels)
    
    # 收集所有测试集样本和标签
    test_samples = []
    test_labels = []
    
    for label_id in tqdm(valid_labels, desc="加载测试集数据"):
        file_path = test_sampler.get_file_path(label_id)
        if file_path:
            samples = np.load(file_path)
            labels = np.ones(len(samples)) * label_id
            test_samples.append(samples)
            test_labels.append(labels)
    
    # 收集所有验证集样本和标签
    val_samples = []
    val_labels = []
    
    for label_id in tqdm(valid_labels, desc="加载验证集数据"):
        file_path = val_sampler.get_file_path(label_id)
        if file_path:
            samples = np.load(file_path)
            labels = np.ones(len(samples)) * label_id
            val_samples.append(samples)
            val_labels.append(labels)
    
    # 合并各自的数据
    train_samples = np.vstack(train_samples) if train_samples else np.array([])
    train_labels = np.concatenate(train_labels) if train_labels else np.array([])
    test_samples = np.vstack(test_samples) if test_samples else np.array([])
    test_labels = np.concatenate(test_labels) if test_labels else np.array([])
    val_samples = np.vstack(val_samples) if val_samples else np.array([])
    val_labels = np.concatenate(val_labels) if val_labels else np.array([])
    
    # 打印数据集统计信息
    print(f"\n{'='*60}\n多分类数据集统计信息\n{'='*60}")
    print(f"训练集: {len(train_labels)} 个样本")
    print(f"测试集: {len(test_labels)} 个样本")
    print(f"验证集: {len(val_labels)} 个样本")
    
    # 类别分布统计
    print("\n类别分布:")
    for label_id in valid_labels:
        train_count = np.sum(train_labels == label_id)
        test_count = np.sum(test_labels == label_id)
        val_count = np.sum(val_labels == label_id)
        total_count = train_count + test_count + val_count
        print(f"标签 {label_id}: 训练集 {train_count}, 测试集 {test_count}, 验证集 {val_count}, 总计 {total_count}")
    
    # 应用PCA（如果需要）
    if apply_pca_flag:
        # 合并所有数据进行PCA拟合
        all_samples = np.vstack([train_samples, test_samples, val_samples])
        
        if n_components == 0:
            # 自动选择主成分数量
            n_components, _, _ = analyze_pca_variance(all_samples, plot=True)
        
        # 创建并拟合PCA模型
        pca_model = PCA(n_components=n_components)
        pca_model.fit(all_samples)
        
        # 应用PCA变换
        train_samples = pca_model.transform(train_samples)
        test_samples = pca_model.transform(test_samples)
        val_samples = pca_model.transform(val_samples)
        
        # 应用标准化（如果需要）
        if norm:
            # 基于所有样本计算归一化参数
            all_transformed = np.vstack([train_samples, test_samples, val_samples])
            mins = np.min(all_transformed, axis=0)
            maxs = np.max(all_transformed, axis=0)
            ranges = maxs - mins + 1e-10  # 避免除零
            
            # 应用归一化
            train_samples = (train_samples - mins) / ranges
            test_samples = (test_samples - mins) / ranges
            val_samples = (val_samples - mins) / ranges
        
        feature_dim = train_samples.shape[1]
        print(f"\n特征维度: {feature_dim} (PCA降维后)")
    else:
        feature_dim = train_samples.shape[1]
        print(f"\n特征维度: {feature_dim} (原始特征)")
        pca_model = None
    
    return {
        'train_samples': train_samples,
        'train_labels': train_labels,
        'test_samples': test_samples,
        'test_labels': test_labels,
        'val_samples': val_samples,
        'val_labels': val_labels,
        'feature_dim': feature_dim,
        'pca_model': pca_model,
        'valid_labels': valid_labels
    }

In [ ]:
def create_mlp_model(trial, input_dim, num_classes, param_space=None):
    """
    使用Optuna trial创建MLP模型
    
    参数:
        trial: Optuna trial对象
        input_dim: 输入特征维度
        num_classes: 类别数量
        param_space: 参数空间字典
    
    返回:
        model: MLP模型
    """
    if param_space is None:
        # 默认参数空间
        param_space = {
            'dropout_rate': (0.3, 0.7),
            'activation': ['relu', 'gelu', 'swish'],
            'layer_sizes': [
                [4096, 4096, 4096, 4096],  # 标准4x4096网络
                [3072, 3072, 3072, 3072],  # 更小的网络
                [2048, 4096, 4096, 2048]   # 钟形网络
            ]
        }
    
    # 从参数空间采样
    dropout_rate = trial.suggest_float('dropout_rate', *param_space['dropout_rate'])
    activation = trial.suggest_categorical('activation', param_space['activation'])
    layer_sizes_idx = trial.suggest_int('layer_sizes_idx', 0, len(param_space['layer_sizes'])-1)
    hidden_dims = param_space['layer_sizes'][layer_sizes_idx]
    
    # 创建模型
    model = BrainVoxelMLP(
        input_dim=input_dim,
        hidden_dims=hidden_dims,
        num_classes=num_classes,
        dropout_rate=dropout_rate,
        activation=activation
    )
    
    return model

def objective(trial, data_loaders, input_dim, num_classes, device, param_space=None):
    """
    Optuna优化目标函数
    
    参数:
        trial: Optuna trial对象
        data_loaders: 包含训练和验证数据加载器的字典
        input_dim: 输入特征维度
        num_classes: 类别数量
        device: 计算设备
        param_space: 参数空间字典
    
    返回:
        val_f1_macro: 验证集F1宏平均分数
    """
    # 训练参数
    if param_space is None:
        param_space = {
            'learning_rate': (1e-6, 1e-3, 'log'),
            'batch_size': [64, 128, 256, 512],
            'weight_decay': (1e-6, 1e-3, 'log'),
            'dropout_rate': (0.3, 0.7),
            'activation': ['relu', 'gelu', 'swish'],
            'optimizer': ['adam', 'adamw'],
            'lr_scheduler': ['cosine', 'step', 'none'],
            'layer_sizes': [
                [4096, 4096, 4096, 4096],  # 标准4x4096网络
                [3072, 3072, 3072, 3072],  # 更小的网络
                [2048, 4096, 4096, 2048]   # 钟形网络
            ]
        }
    
    # 从参数空间采样
    learning_rate = trial.suggest_float('learning_rate', *param_space['learning_rate'][:2], log=True)
    weight_decay = trial.suggest_float('weight_decay', *param_space['weight_decay'][:2], log=True)
    optimizer_name = trial.suggest_categorical('optimizer', param_space['optimizer'])
    lr_scheduler_type = trial.suggest_categorical('lr_scheduler', param_space['lr_scheduler'])
    
    # 创建模型
    model = create_mlp_model(trial, input_dim, num_classes, param_space)
    model = model.to(device)
    
    # 创建优化器
    if optimizer_name == 'adam':
        optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    elif optimizer_name == 'adamw':
        optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    else:
        optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    
    # 创建学习率调度器
    lr_scheduler = None
    if lr_scheduler_type == 'cosine':
        lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)
    elif lr_scheduler_type == 'step':
        step_size = trial.suggest_int('lr_step_size', 5, 15)
        gamma = trial.suggest_float('lr_gamma', 0.1, 0.5)
        lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=step_size, gamma=gamma)
    
    # 定义损失函数 - 使用交叉熵
    criterion = nn.CrossEntropyLoss(ignore_index=-1)
    
    # 训练模型 - 简化版本，只训练几个epoch用于评估
    num_epochs = 10  # 贝叶斯优化时使用较少的epoch
    val_f1_values = []
    
    # 训练循环
    for epoch in range(num_epochs):
        model.train()
        for batch_idx, (data, target) in enumerate(data_loaders['train']):
            data, target = data.to(device), target.to(device)
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()
        
        # 验证
        model.eval()
        all_preds = []
        all_targets = []
        
        with torch.no_grad():
            for data, target in data_loaders['val']:
                data, target = data.to(device), target.to(device)
                output = model(data)
                _, preds = torch.max(output, 1)
                
                # 只评估非背景像素
                valid_mask = target != -1
                all_preds.extend(preds[valid_mask].cpu().numpy())
                all_targets.extend(target[valid_mask].cpu().numpy())
        
        # 计算F1分数
        val_f1_macro = f1_score(all_targets, all_preds, average='macro')
        val_f1_values.append(val_f1_macro)
        
        # 更新学习率
        if lr_scheduler is not None:
            lr_scheduler.step()
        
        # 保存更好的结果
        trial.report(val_f1_macro, epoch)
        
        # 处理提前停止
        if trial.should_prune():
            raise optuna.TrialPruned()
    
    # 返回最佳F1分数
    return max(val_f1_values)

def run_bayesian_optimization(data_loaders, input_dim, num_classes, device, param_space=None, n_trials=30, study_name="mlp_optimization", save_path="./results"):
    """
    运行贝叶斯优化
    
    参数:
        data_loaders: 包含训练和验证数据加载器的字典
        input_dim: 输入特征维度
        num_classes: 类别数量
        device: 计算设备
        param_space: 参数空间字典
        n_trials: 优化试验次数
        study_name: 研究名称
        save_path: 结果保存路径
    
    返回:
        study: Optuna study对象
        best_params: 最佳参数
    """
    # 创建研究
    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED),
        pruner=optuna.pruners.MedianPruner(n_warmup_steps=5),
        study_name=study_name
    )
    
    # 运行优化
    study.optimize(
        lambda trial: objective(trial, data_loaders, input_dim, num_classes, device, param_space),
        
        n_trials=n_trials
    )
    
    # 打印优化结果
    print("贝叶斯优化完成.")
    print(f"最佳F1分数: {study.best_value:.4f}")
    print("最佳参数:")
    for key, value in study.best_params.items():
        print(f"  {key}: {value}")
    
    # 保存结果
    if not os.path.exists(save_path):
        os.makedirs(save_path)
    
    # 保存优化结果
    result_path = os.path.join(save_path, f"{study_name}_results.json")
    with open(result_path, 'w') as f:
        json.dump({
            'best_params': study.best_params,
            'best_value': study.best_value,
            'all_trials': [
                {
                    'number': t.number,
                    'params': t.params,
                    'value': t.value if t.value is not None else None,
                    'state': t.state.name
                }
                for t in study.trials
            ]
        }, f, indent=2)
    
    # 生成可视化 (如果可用)
    try:
        # 参数重要性图
        param_importance_fig = plot_param_importances(study)
        param_importance_fig.write_image(os.path.join(save_path, f"{study_name}_param_importance.png"))
        
        # 优化历史图
        optimization_history_fig = plot_optimization_history(study)
        optimization_history_fig.write_image(os.path.join(save_path, f"{study_name}_optimization_history.png"))
    except Exception as e:
        print(f"可视化生成失败: {e}")
    
    return study, study.best_params

In [ ]:
def visualize_dataset_distribution(dataset_dict, label_id, save_path=None):
    """可视化数据集的分布情况"""
    plt.figure(figsize=(15, 5))
    
    # 1. 1. 正负样本比例图
    plt.subplot(1, 3, 1)
    datasets = ['Training Set', 'Test Set', 'Validation Set']
    pos_counts = [
        np.sum(dataset_dict['train_labels'] == 1),
        np.sum(dataset_dict['test_labels'] == 1),
        np.sum(dataset_dict['val_labels'] == 1)
    ]
    neg_counts = [
        np.sum(dataset_dict['train_labels'] == 0),
        np.sum(dataset_dict['test_labels'] == 0),
        np.sum(dataset_dict['val_labels'] == 0)
    ]
    
    x = np.arange(len(datasets))
    width = 0.35
    
    plt.bar(x - width/2, pos_counts, width, label='Positive Samples')
    plt.bar(x + width/2, neg_counts, width, label='Negative Samples')
    
    plt.xlabel('Dataset')
    plt.ylabel('Sample Count')
    plt.title(f'Positive and Negative Sample Distribution for Label {label_id}')
    plt.xticks(x, datasets)
    plt.legend()
    
    # 2. 正负比例饼图
    plt.subplot(1, 3, 2)
    total_pos = sum(pos_counts)
    total_neg = sum(neg_counts)
    plt.pie([total_pos, total_neg], labels=['Positive Samples', 'Negative Samples'], 
            autopct='%1.1f%%', startangle=90)
    plt.axis('equal')
    plt.title('Positive vs Negative Sample Proportion')
    
    # 3. 数据集大小比较
    plt.subplot(1, 3, 3)
    set_sizes = [
        len(dataset_dict['train_labels']),
        len(dataset_dict['test_labels']),
        len(dataset_dict['val_labels'])
    ]
    plt.pie(set_sizes, labels=datasets, autopct='%1.1f%%', startangle=90)
    plt.axis('equal')
    plt.title('Dataset Size Distribution')
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path)
    plt.show()


In [ ]:
# 定义数据目录路径
DATA_DIRS = {
    'train_dir': "/home/jovyan/gpu_space/workspace_jiayi/KAN training/brain_voxel_data/restructured/train",
    'test_dir': "/home/jovyan/gpu_space/workspace_jiayi/KAN training/brain_voxel_data/restructured/test",
    'val_dir': "/home/jovyan/gpu_space/workspace_jiayi/KAN training/brain_voxel_data/restructured/val"
}

# 载入多分类数据集 - 不进行类别平衡
print("开始加载脑体素数据集...")
dataset_dict = load_multiclass_data(
    DATA_DIRS, 
    apply_pca_flag=APPLY_PCA, 
    n_components=N_PCA, 
    norm=NORM
)
print(f"数据加载完成! 共载入 {len(dataset_dict['train_labels'])} 个训练样本，{len(dataset_dict['val_labels'])} 个验证样本，{len(dataset_dict['test_labels'])} 个测试样本")
print(f"特征维度: {dataset_dict['feature_dim']}")

In [ ]:
# 创建训练、测试和验证数据集
print("创建数据集和数据加载器...")
train_dataset = BrainVoxelDataset(dataset_dict['train_samples'], dataset_dict['train_labels'])
test_dataset = BrainVoxelDataset(dataset_dict['test_samples'], dataset_dict['test_labels'])
val_dataset = BrainVoxelDataset(dataset_dict['val_samples'], dataset_dict['val_labels'])

# 创建数据加载器
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

# 将数据加载器打包为字典
data_loaders = {
    'train': train_loader,
    'val': val_loader
}

# 设置计算设备
device = torch.device(f"cuda:{DEVICE}" if DEVICE>=0 and torch.cuda.is_available() else "cpu")
print(f"使用设备: {device}")
print(f"训练批次数: {len(train_loader)}，验证批次数: {len(val_loader)}，测试批次数: {len(test_loader)}")

In [ ]:
# 定义基准MLP模型参数空间
print("\n定义贝叶斯优化参数空间...")
baseline_param_space = {
    'learning_rate': (1e-6, 1e-3, 'log'),
    'batch_size': [64, 128, 256, 512],
    'weight_decay': (1e-6, 1e-3, 'log'),
    'dropout_rate': (0.3, 0.7),
    'activation': ['relu', 'gelu', 'swish'],
    'optimizer': ['adam', 'adamw'],
    'lr_scheduler': ['cosine', 'step', 'none'],
    'layer_sizes': [
        [4096, 4096, 4096, 4096]  # 标准4x4096网络 - 基准
    ]
}

print("参数空间详情:")
for param, value in baseline_param_space.items():
    print(f"  {param}: {value}")

# 运行贝叶斯优化查找基准MLP的最优超参数
print("\n开始进行贝叶斯优化，寻找最优超参数配置...")
print(f"优化试验次数: {30}，优化指标: 验证集F1宏平均分数")

study, best_params = run_bayesian_optimization(
    data_loaders=data_loaders,
    input_dim=dataset_dict['feature_dim'],
    num_classes=NUM_CLASS,
    device=device,
    param_space=baseline_param_space,
    n_trials=30,  # 实际应用中可能需要50-100次试验
    study_name="mlp_baseline_optimization",
    save_path=SAVE_PATH
)

print("\n贝叶斯优化完成!")
print(f"最优F1分数: {study.best_value:.4f}")
print("最优超参数配置:")
for param, value in best_params.items():
    print(f"  {param}: {value}")

In [ ]:
# 使用最佳参数创建模型
print("\n使用最优超参数创建基准MLP模型...")
best_dropout_rate = best_params.get('dropout_rate', 0.5)
best_activation = best_params.get('activation', 'relu')
layer_sizes_idx = best_params.get('layer_sizes_idx', 0)
best_layer_sizes = baseline_param_space['layer_sizes'][layer_sizes_idx]

# 创建最优基准模型
best_model = BrainVoxelMLP(
    input_dim=dataset_dict['feature_dim'],
    hidden_dims=best_layer_sizes,
    num_classes=NUM_CLASS,
    dropout_rate=best_dropout_rate,
    activation=best_activation
).to(device)

# 计算类别权重（可选）
class_weights = calculate_class_weights(dataset_dict['train_labels']).to(device)

# 定义损失函数和优化器
best_lr = best_params.get('learning_rate', 1e-5)
best_weight_decay = best_params.get('weight_decay', 1e-5)
best_optimizer = best_params.get('optimizer', 'adam')

if best_optimizer == 'adam':
    optimizer = torch.optim.Adam(best_model.parameters(), lr=best_lr, weight_decay=best_weight_decay)
else:
    optimizer = torch.optim.AdamW(best_model.parameters(), lr=best_lr, weight_decay=best_weight_decay)

# 定义损失函数 - 使用类别权重来处理不平衡问题
criterion = nn.CrossEntropyLoss(weight=class_weights, ignore_index=-1)

# 配置学习率调度器
best_lr_scheduler = best_params.get('lr_scheduler', 'cosine')
lr_scheduler = None

if best_lr_scheduler == 'cosine':
    lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCH)
elif best_lr_scheduler == 'step':
    best_step_size = best_params.get('lr_step_size', 10)
    best_gamma = best_params.get('lr_gamma', 0.5)
    lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=best_step_size, gamma=best_gamma)

# 打印模型结构
print("\n模型结构摘要:")
summary(best_model)

# 保存最优超参数配置
with open(os.path.join(SAVE_PATH, 'best_params_baseline.json'), 'w') as f:
    json.dump(best_params, f, indent=4)
print(f"最优超参数配置已保存至: {os.path.join(SAVE_PATH, 'best_params_baseline.json')}")

In [ ]:
# 使用最优超参数训练模型
print("\n开始使用最优超参数训练基准MLP模型...")
print(f"训练轮数: {EPOCH}, 验证频率: 每{VAL_EPOCH}个epoch")
print(f"学习率: {best_lr}, 权重衰减: {best_weight_decay}, Dropout率: {best_dropout_rate}")
print(f"优化器: {best_optimizer}, 学习率调度: {best_lr_scheduler}")
print(f"批量大小: {BATCH_SIZE}, 层大小: {best_layer_sizes}")
print(f"激活函数: {best_activation}")

training_start_time = time.time()
training_results = train_brain_voxel_mlp_multiclass(
    model=best_model,
    train_loader=train_loader,
    val_loader=val_loader, 
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    num_epochs=EPOCH,
    val_epoch=VAL_EPOCH,
    save_path=SAVE_PATH,
    lr_scheduler=lr_scheduler
)
training_time = time.time() - training_start_time

print(f"\n模型训练完成! 耗时: {training_time:.2f}秒")
print(f"最终训练损失: {training_results['loss_list'][-1]:.4f}")
print(f"最佳验证F1分数: {max(training_results['val_f1_macro_list']):.4f}")

# 获取最佳模型
best_model_path = get_best_model(
    training_results['val_f1_macro_list'],  # 使用宏平均F1作为选择标准
    training_results['val_epoch_list'],
    SAVE_PATH,
    metric='f1'
)
print(f"最佳模型已保存: {os.path.basename(best_model_path)}")

In [ ]:
# 加载最佳模型
print("\n加载最佳模型进行评估...")
best_model = BrainVoxelMLP(
    input_dim=dataset_dict['feature_dim'],
    hidden_dims=best_layer_sizes,
    num_classes=NUM_CLASS,
    dropout_rate=best_dropout_rate,
    activation=best_activation
).to(device)
best_model.load_state_dict(torch.load(best_model_path)['state_dict'])

# 在测试集上评估最佳模型
print("\n在测试集上评估最佳基准MLP模型...")
eval_start_time = time.time()
test_results = evaluate_model_multiclass(best_model, test_loader, device)
eval_time = time.time() - eval_start_time
print(f"评估完成! 耗时: {eval_time:.2f}秒")

# 打印主要评估指标
print("\n测试集评估结果摘要:")
print(f"准确率:       {test_results['accuracy']:.4f}")
print(f"平衡准确率:   {test_results['balanced_accuracy']:.4f}")
print(f"宏平均F1:     {test_results['f1_macro']:.4f}")
print(f"加权F1:       {test_results['f1_weighted']:.4f}")
print(f"Kappa系数:    {test_results['kappa']:.4f}")
print(f"类别数量:     {len(np.unique(test_results['targets']))}")
print(f"总样本数:     {len(test_results['targets'])}")

# 打印分类报告
print("\n分类报告:")
print(test_results['report'])

In [ ]:
# 可视化训练过程
print("\n生成训练过程可视化...")

plt.figure(figsize=(15, 5))

# 绘制损失曲线
plt.subplot(1, 3, 1)
plt.plot(range(len(training_results['loss_list'])), training_results['loss_list'])
plt.title('Training Loss Curve')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(True)

# 绘制准确率曲线
plt.subplot(1, 3, 2)
plt.plot(range(len(training_results['acc_list'])), training_results['acc_list'], label='Training Accuracy')
plt.plot(training_results['val_epoch_list'], training_results['val_acc_list'], label='Validation Accuracy')
plt.title('Accuracy Comparison')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

# 绘制F1和Kappa曲线
plt.subplot(1, 3, 3)
plt.plot(training_results['val_epoch_list'], training_results['val_f1_macro_list'], 'g-', label='F1 Macro')
plt.plot(training_results['val_epoch_list'], training_results['val_kappa_list'], 'r--', label='Kappa Coefficient')
plt.title('Validation Metrics')
plt.xlabel('Epoch')
plt.ylabel('Score')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_PATH, 'baseline_mlp_training_curves.png'))
plt.show()

print(f"训练曲线已保存至: {os.path.join(SAVE_PATH, 'baseline_mlp_training_curves.png')}")

# 额外可视化 - 类别性能对比
plt.figure(figsize=(12, 6))
# 获取每个类别的F1分数
class_f1 = test_results['class_f1']
# 排序后显示前20个和后20个类别
sorted_indices = np.argsort(class_f1)
worst_classes = sorted_indices[:20]
best_classes = sorted_indices[-20:]

plt.subplot(1, 2, 1)
plt.barh(range(len(worst_classes)), class_f1[worst_classes])
plt.yticks(range(len(worst_classes)), [f"Class {i}" for i in worst_classes])
plt.xlabel('F1 Score')
plt.title('20 Classes with Lowest F1 Scores')
plt.grid(True, axis='x')

plt.subplot(1, 2, 2)
plt.barh(range(len(best_classes)), class_f1[best_classes])
plt.yticks(range(len(best_classes)), [f"Class {i}" for i in best_classes])
plt.xlabel('F1 Score')
plt.title('20 Classes with Highest F1 Scores')
plt.grid(True, axis='x')

plt.tight_layout()
plt.savefig(os.path.join(SAVE_PATH, 'class_f1_comparison.png'))
plt.show()

print(f"类别F1分数对比图已保存至: {os.path.join(SAVE_PATH, 'class_f1_comparison.png')}")
print("\n基准MLP模型的训练和评估全部完成!")

In [ ]:
print(model)


In [ ]:
# 定义不同的MLP架构变体
architecture_variants = {
    "variant1_narrow_to_wide": {
        'name': "窄变宽型MLP",
        'param_space': {
            'learning_rate': (1e-6, 1e-3, 'log'),
            'batch_size': [64, 128, 256, 512],
            'weight_decay': (1e-6, 1e-3, 'log'),
            'dropout_rate': (0.2, 0.6),
            'activation': ['relu', 'gelu', 'swish'],
            'optimizer': ['adam', 'adamw'],
            'lr_scheduler': ['cosine', 'step', 'none'],
            'layer_sizes': [
                [512, 1024, 2048, 4096],  # 标准窄变宽
                [768, 1536, 3072, 4096],  # 更宽的中间层
                [512, 1024, 2048, 3072]   # 较小的最后一层
            ]
        }
    },
    
    "variant2_wide_to_narrow": {
        'name': "宽变窄型MLP",
        'param_space': {
            'learning_rate': (1e-6, 1e-3, 'log'),
            'batch_size': [64, 128, 256, 512],
            'weight_decay': (1e-6, 1e-3, 'log'),
            'dropout_rate': (0.2, 0.6),
            'activation': ['relu', 'gelu', 'swish'],
            'optimizer': ['adam', 'adamw'],
            'lr_scheduler': ['cosine', 'step', 'none'],
            'layer_sizes': [
                [4096, 2048, 1024, 512],  # 标准宽变窄
                [3072, 2048, 1024, 512],  # 调整第一层
                [4096, 3072, 2048, 1024]  # 更缓慢的缩小
            ]
        }
    },
    
    "variant3_bell_shaped": {
        'name': "钟形结构MLP",
        'param_space': {
            'learning_rate': (1e-6, 1e-3, 'log'),
            'batch_size': [64, 128, 256, 512],
            'weight_decay': (1e-6, 1e-3, 'log'),
            'dropout_rate': (0.2, 0.6),
            'activation': ['relu', 'gelu', 'swish'],
            'optimizer': ['adam', 'adamw'],
            'lr_scheduler': ['cosine', 'step', 'none'],
            'layer_sizes': [
                [2048, 4096, 4096, 2048],  # 标准钟形
                [1024, 2048, 2048, 1024],  # 较小钟形
                [3072, 4096, 4096, 3072]   # 较大钟形
            ]
        }
    },
    
    "variant4_deep": {
        'name': "深层MLP",
        'param_space': {
            'learning_rate': (1e-6, 1e-3, 'log'),
            'batch_size': [64, 128, 256, 512],
            'weight_decay': (1e-6, 1e-3, 'log'),
            'dropout_rate': (0.2, 0.6),
            'activation': ['relu', 'gelu', 'swish'],
            'optimizer': ['adam', 'adamw'],
            'lr_scheduler': ['cosine', 'step', 'none'],
            'layer_sizes': [
                [2048, 2048, 2048, 2048, 2048, 2048],  # 6层2048
                [1024, 2048, 3072, 3072, 2048, 1024],  # 6层变化
                [2048, 2048, 2048, 2048, 2048]         # 5层2048
            ]
        }
    }


}

# 为每个架构变体创建自定义MLP模型类
class DeepMLP(BrainVoxelMLP):
    """
    深层MLP变体，支持不同数量的层
    """
    def __init__(self, input_dim, hidden_dims, num_classes, dropout_rate=0.5, activation='relu'):
        super(DeepMLP, self).__init__(input_dim, hidden_dims, num_classes, dropout_rate, activation)
        
        # 重新实现以支持任意长度的hidden_dims
        self.layers = nn.ModuleList()
        
        # 选择激活函数
        if activation == 'relu':
            act_fn = nn.ReLU()
        elif activation == 'gelu':
            act_fn = nn.GELU()
        elif activation == 'swish':
            act_fn = nn.SiLU()  # PyTorch中的SiLU就是Swish激活函数
        else:
            act_fn = nn.ReLU()  # 默认使用ReLU
        
        # 输入层到第一个隐藏层
        self.layers.append(nn.Linear(input_dim, hidden_dims[0]))
        self.layers.append(act_fn)
        self.layers.append(nn.Dropout(dropout_rate))
        
        # 添加中间隐藏层
        for i in range(len(hidden_dims) - 1):
            self.layers.append(nn.Linear(hidden_dims[i], hidden_dims[i+1]))
            self.layers.append(act_fn)
            self.layers.append(nn.Dropout(dropout_rate))
        
        # 最后的分类层
        self.layers.append(nn.Linear(hidden_dims[-1], num_classes))

# 优化架构变体
variant_results = {}

for variant_id, variant_info in architecture_variants.items():
    print(f"\n{'='*50}")
    print(f"开始优化架构变体: {variant_info['name']}")
    print(f"{'='*50}")
    
    # 使用自定义的优化空间
    param_space = variant_info['param_space']
    
    # 如果是深层MLP变体，使用DeepMLP类
    model_class = DeepMLP if variant_id == "variant4_deep" else BrainVoxelMLP
    
    # 设置variant特定的目标函数
    def variant_objective(trial, variant_id=None):
        """
        Optuna优化目标函数 - 针对架构变体

        参数:
            trial: Optuna trial对象
            variant_id: 架构变体ID
        
        返回:
            val_f1_macro: 验证集F1宏平均分数
        """
        # 获取当前变体的参数空间
        if variant_id is None or variant_id not in architecture_variants:
            # 默认使用基准MLP的参数空间
            param_space = baseline_param_space
        else:
            param_space = architecture_variants[variant_id]['param_space']

        # 从参数空间采样
        learning_rate = trial.suggest_float('learning_rate', *param_space['learning_rate'][:2], log=True)
        weight_decay = trial.suggest_float('weight_decay', *param_space['weight_decay'][:2], log=True)
        dropout_rate = trial.suggest_float('dropout_rate', *param_space['dropout_rate'])
        activation = trial.suggest_categorical('activation', param_space['activation'])
        optimizer_name = trial.suggest_categorical('optimizer', param_space['optimizer'])
        lr_scheduler_type = trial.suggest_categorical('lr_scheduler', param_space['lr_scheduler'])
        layer_sizes_idx = trial.suggest_int('layer_sizes_idx', 0, len(param_space['layer_sizes'])-1)
        hidden_dims = param_space['layer_sizes'][layer_sizes_idx]
        
        # 创建模型 - 根据变体ID选择正确的模型类
        if variant_id == "variant5_residual":
            model = ResidualBrainVoxelMLP(
                input_dim=dataset_dict['feature_dim'],
                hidden_dims=hidden_dims,
                num_classes=NUM_CLASS,
                dropout_rate=dropout_rate,
                activation=activation
            ).to(device)
        elif variant_id == "variant4_deep":
            model = DeepMLP(
                input_dim=dataset_dict['feature_dim'],
                hidden_dims=hidden_dims,
                num_classes=NUM_CLASS,
                dropout_rate=dropout_rate,
                activation=activation
            ).to(device)
        else:
            model = BrainVoxelMLP(
                input_dim=dataset_dict['feature_dim'],
                hidden_dims=hidden_dims,
                num_classes=NUM_CLASS,
                dropout_rate=dropout_rate,
                activation=activation
            ).to(device)
        
        # 创建优化器
        if optimizer_name == 'adam':
            optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
        elif optimizer_name == 'adamw':
            optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
        else:
            optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
        
        # 创建学习率调度器
        lr_scheduler = None
        if lr_scheduler_type == 'cosine':
            lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)
        elif lr_scheduler_type == 'step':
            step_size = trial.suggest_int('lr_step_size', 5, 15)
            gamma = trial.suggest_float('lr_gamma', 0.1, 0.5)
            lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=step_size, gamma=gamma)
        
        # 定义损失函数
        class_weights = calculate_class_weights(dataset_dict['train_labels']).to(device)
        criterion = nn.CrossEntropyLoss(weight=class_weights, ignore_index=-1)
        
        # 训练模型 - 简化版本，只训练几个epoch用于评估
        num_epochs = 10  # 贝叶斯优化时使用较少的epoch
        val_f1_values = []
        
        # 训练循环
        for epoch in range(num_epochs):
            model.train()
            for batch_idx, (data, target) in enumerate(train_loader):
                data, target = data.to(device), target.to(device)
                optimizer.zero_grad()
                output = model(data)
                loss = criterion(output, target)
                loss.backward()
                optimizer.step()
            
            # 验证
            model.eval()
            all_preds = []
            all_targets = []
            
            with torch.no_grad():
                for data, target in val_loader:
                    data, target = data.to(device), target.to(device)
                    output = model(data)
                    _, preds = torch.max(output, 1)
                    
                    # 只评估非背景像素
                    valid_mask = target != -1
                    all_preds.extend(preds[valid_mask].cpu().numpy())
                    all_targets.extend(target[valid_mask].cpu().numpy())
            
            # 计算F1分数
            val_f1_macro = f1_score(all_targets, all_preds, average='macro')
            val_f1_values.append(val_f1_macro)
            
            # 更新学习率
            if lr_scheduler is not None:
                lr_scheduler.step()
            
            # 保存更好的结果
            trial.report(val_f1_macro, epoch)
            
            # 处理提前停止
            if trial.should_prune():
                raise optuna.TrialPruned()
        
        # 返回最佳F1分数
        return max(val_f1_values)


    
    # 创建变体专用目录
    variant_save_path = os.path.join(SAVE_PATH, variant_id)
    if not os.path.exists(variant_save_path):
        os.makedirs(variant_save_path)
    
    # 创建研究
    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED),
        pruner=optuna.pruners.MedianPruner(n_warmup_steps=5),
        study_name=f"{variant_id}_optimization"
    )
        
    study.optimize(
        lambda trial: variant_objective(trial, variant_id=variant_id), 
        n_trials=15
    )  # 每个变体使用15次试验

    
    # 保存结果
    best_params = study.best_params
    
    # 保存优化结果
    result_path = os.path.join(variant_save_path, "optimization_results.json")
    with open(result_path, 'w') as f:
        json.dump({
            'variant_name': variant_info['name'],
            'best_params': best_params,
            'best_value': study.best_value,
            'all_trials': [
                {
                    'number': t.number,
                    'params': t.params,
                    'value': t.value if t.value is not None else None,
                    'state': t.state.name
                }
                for t in study.trials
            ]
        }, f, indent=2)
    
    # 存储结果用于比较
    variant_results[variant_id] = {
        'name': variant_info['name'],
        'best_params': best_params,
        'best_f1': study.best_value
    }
    
    print(f"完成 {variant_info['name']} 的优化")
    print(f"最佳F1分数: {study.best_value:.4f}")
    print("最佳参数:")
    for key, value in best_params.items():
        print(f"  {key}: {value}")

# 比较所有架构变体的性能
print("\n架构变体比较:")
variants_df = pd.DataFrame([
    {
        'Variant ID': variant_id,
        'Name': info['name'],
        'Best F1': info['best_f1'],
        'Learning Rate': info['best_params'].get('learning_rate'),
        'Dropout Rate': info['best_params'].get('dropout_rate'),
        'Activation': info['best_params'].get('activation'),
        'Layer Config': param_space['layer_sizes'][info['best_params'].get('layer_sizes_idx', 0)]
    }
    for variant_id, info in variant_results.items()
])

# 按最佳F1分数排序
variants_df = variants_df.sort_values('Best F1', ascending=False)
print(variants_df)

# 可视化比较结果
plt.figure(figsize=(10, 6))
sns.barplot(x='Variant ID', y='Best F1', data=variants_df)
plt.title('架构变体性能比较')
plt.ylabel('验证集F1宏平均分数')
plt.grid(True, axis='y')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_PATH, 'architecture_variants_comparison.png'))
plt.show()

In [ ]:
# 残差连接全连接网络实现
class ResidualBrainVoxelMLP(nn.Module):
    """
    带残差连接的脑体素MLP网络
    """
    def __init__(self, input_dim, hidden_dims, num_classes, dropout_rate=0.5, activation='relu'):
        """
        初始化模型
        
        参数:
            input_dim: 输入特征维度
            hidden_dims: 隐藏层维度列表，例如[4096, 4096, 4096, 4096]
            num_classes: 类别数量 (102)
            dropout_rate: Dropout比率
            activation: 激活函数，支持'relu'和'gelu'
        """
        super(ResidualBrainVoxelMLP, self).__init__()
        
        # 选择激活函数
        if activation == 'relu':
            self.act_fn = nn.ReLU()
        elif activation == 'gelu':
            self.act_fn = nn.GELU()
        elif activation == 'swish':
            self.act_fn = nn.SiLU()  # PyTorch中的SiLU就是Swish激活函数
        else:
            self.act_fn = nn.ReLU()  # 默认使用ReLU
        
        # 输入层
        self.input_layer = nn.Linear(input_dim, hidden_dims[0])
        self.input_act = self.act_fn
        self.input_dropout = nn.Dropout(dropout_rate)
        
        # 残差块
        self.residual_blocks = nn.ModuleList()
        
        for i in range(len(hidden_dims) - 1):
            # 判断是否需要维度转换
            if hidden_dims[i] != hidden_dims[i+1]:
                shortcut = nn.Linear(hidden_dims[i], hidden_dims[i+1])
            else:
                shortcut = nn.Identity()
                
            # 主路径
            main_path = nn.Sequential(
                nn.Linear(hidden_dims[i], hidden_dims[i+1]),
                self.act_fn,
                nn.Dropout(dropout_rate),
                nn.Linear(hidden_dims[i+1], hidden_dims[i+1]),
                nn.Dropout(dropout_rate)
            )
            
            self.residual_blocks.append(nn.ModuleDict({
                'main_path': main_path,
                'shortcut': shortcut
            }))
            
        # 输出层
        self.output_layer = nn.Linear(hidden_dims[-1], num_classes)
    
    def forward(self, x):
        """
        前向传播
        
        参数:
            x: 输入特征，形状为(batch_size, input_dim)
            
        返回:
            output: 模型输出，形状为(batch_size, num_classes)
        """
        # 输入层
        x = self.input_layer(x)
        x = self.input_act(x)
        x = self.input_dropout(x)
        
        # 残差块
        for block in self.residual_blocks:
            identity = x
            x = block['main_path'](x)
            x = block['shortcut'](identity) + x  # 残差连接
            x = self.act_fn(x)  # 残差后的激活函数
        
        # 输出层
        x = self.output_layer(x)
        return x

# 添加残差网络架构变体到现有字典
architecture_variants["variant5_residual"] = {
    'name': "残差连接MLP",
    'param_space': {
        'learning_rate': (1e-6, 1e-3, 'log'),
        'batch_size': [64, 128, 256, 512],
        'weight_decay': (1e-6, 1e-3, 'log'),
        'dropout_rate': (0.2, 0.6),
        'activation': ['relu', 'gelu', 'swish'],
        'optimizer': ['adam', 'adamw'],
        'lr_scheduler': ['cosine', 'step', 'none'],
        'layer_sizes': [
            [4096, 4096, 4096, 4096],  # 标准4x4096
            [2048, 2048, 2048, 2048],  # 更小的网络
            [1024, 2048, 2048, 1024]   # 钟形结构
        ]
    }
}

In [ ]:
# 修改后的最佳模型加载逻辑
# 选择性能最好的架构变体
best_variant_id = variants_df.iloc[0]['Variant ID']
best_variant_info = variant_results[best_variant_id]
best_variant_name = best_variant_info['name']
best_variant_params = best_variant_info['best_params']

print(f"\n选择性能最好的架构变体: {best_variant_name}")
print(f"最佳F1分数: {best_variant_info['best_f1']:.4f}")

# 获取最优参数
best_lr = best_variant_params.get('learning_rate', 1e-5)
best_weight_decay = best_variant_params.get('weight_decay', 1e-5)
best_dropout_rate = best_variant_params.get('dropout_rate', 0.5)
best_activation = best_variant_params.get('activation', 'relu')
best_optimizer_name = best_variant_params.get('optimizer', 'adam')
best_lr_scheduler_type = best_variant_params.get('lr_scheduler', 'cosine')
layer_sizes_idx = best_variant_params.get('layer_sizes_idx', 0)
param_space = architecture_variants[best_variant_id]['param_space']
best_layer_sizes = param_space['layer_sizes'][layer_sizes_idx]

# 创建最优架构模型
if best_variant_id == "variant5_residual":
    final_model = ResidualBrainVoxelMLP(
        input_dim=dataset_dict['feature_dim'],
        hidden_dims=best_layer_sizes,
        num_classes=NUM_CLASS,
        dropout_rate=best_dropout_rate,
        activation=best_activation
    ).to(device)
elif best_variant_id == "variant4_deep":
    final_model = DeepMLP(
        input_dim=dataset_dict['feature_dim'],
        hidden_dims=best_layer_sizes,
        num_classes=NUM_CLASS,
        dropout_rate=best_dropout_rate,
        activation=best_activation
    ).to(device)
else:
    final_model = BrainVoxelMLP(
        input_dim=dataset_dict['feature_dim'],
        hidden_dims=best_layer_sizes,
        num_classes=NUM_CLASS,
        dropout_rate=best_dropout_rate,
        activation=best_activation
    ).to(device)
# 打印模型结构
summary(final_model)

# 定义损失函数和优化器
class_weights = calculate_class_weights(dataset_dict['train_labels']).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights, ignore_index=-1)

if best_optimizer_name == 'adam':
    optimizer = torch.optim.Adam(final_model.parameters(), lr=best_lr, weight_decay=best_weight_decay)
else:
    optimizer = torch.optim.AdamW(final_model.parameters(), lr=best_lr, weight_decay=best_weight_decay)

# 配置学习率调度器
lr_scheduler = None
if best_lr_scheduler_type == 'cosine':
    lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCH)
elif best_lr_scheduler_type == 'step':
    best_step_size = best_variant_params.get('lr_step_size', 10)
    best_gamma = best_variant_params.get('lr_gamma', 0.5)
    lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=best_step_size, gamma=best_gamma)

# 保存最优架构和超参数
final_config = {
    'model_name': f"BrainVoxel_{best_variant_name}",
    'architecture_variant': best_variant_id,
    'best_params': best_variant_params,
    'layer_sizes': best_layer_sizes,
    'feature_dim': dataset_dict['feature_dim'],
    'num_classes': NUM_CLASS
}

with open(os.path.join(SAVE_PATH, 'final_best_model_config.json'), 'w') as f:
    json.dump(final_config, f, indent=4)

# 完整训练最优架构
print("\n开始完整训练最优架构模型...")
final_training_results = train_brain_voxel_mlp_multiclass(
    model=final_model,
    train_loader=train_loader,
    val_loader=val_loader, 
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    num_epochs=EPOCH,
    val_epoch=VAL_EPOCH,
    save_path=SAVE_PATH,
    lr_scheduler=lr_scheduler
)
# 获取最佳模型
final_best_model_path = get_best_model(
    final_training_results['val_f1_macro_list'],
    final_training_results['val_epoch_list'],
    SAVE_PATH,
    metric='f1'
)

# 加载最佳模型
if best_variant_id == "variant5_residual":
    final_best_model = ResidualBrainVoxelMLP(
        input_dim=dataset_dict['feature_dim'],
        hidden_dims=best_layer_sizes,
        num_classes=NUM_CLASS,
        dropout_rate=best_dropout_rate,
        activation=best_activation
    ).to(device)
elif best_variant_id == "variant4_deep":
    final_best_model = DeepMLP(
        input_dim=dataset_dict['feature_dim'],
        hidden_dims=best_layer_sizes,
        num_classes=NUM_CLASS,
        dropout_rate=best_dropout_rate,
        activation=best_activation
    ).to(device)
else:
    final_best_model = BrainVoxelMLP(
        input_dim=dataset_dict['feature_dim'],
        hidden_dims=best_layer_sizes,
        num_classes=NUM_CLASS,
        dropout_rate=best_dropout_rate,
        activation=best_activation
    ).to(device)

final_best_model.load_state_dict(torch.load(final_best_model_path)['state_dict'])

# 在测试集上评估最佳模型
print("\n在测试集上评估最优架构模型...")
final_test_results = evaluate_model_multiclass(final_best_model, test_loader, device)

# 打印主要评估指标
print(f"测试集准确率: {final_test_results['accuracy']:.4f}")
print(f"测试集平衡准确率: {final_test_results['balanced_accuracy']:.4f}")
print(f"测试集宏平均F1: {final_test_results['f1_macro']:.4f}")
print(f"测试集加权F1: {final_test_results['f1_weighted']:.4f}")
print(f"测试集Kappa系数: {final_test_results['kappa']:.4f}")

# 打印分类报告
print("\n分类报告:")
print(final_test_results['report'])

# 可视化训练过程
plt.figure(figsize=(15, 5))

# 绘制损失曲线
plt.subplot(1, 3, 1)
plt.plot(range(len(final_training_results['loss_list'])), final_training_results['loss_list'])
plt.title('Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(True)

# 绘制准确率曲线
plt.subplot(1, 3, 2)
plt.plot(range(len(final_training_results['acc_list'])), final_training_results['acc_list'], label='Train Acc')
plt.plot(final_training_results['val_epoch_list'], final_training_results['val_acc_list'], label='Val Acc')
plt.title('Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

# 绘制F1和Kappa曲线
plt.subplot(1, 3, 3)
plt.plot(final_training_results['val_epoch_list'], final_training_results['val_f1_macro_list'], 'g-', label='F1 Macro')
plt.plot(final_training_results['val_epoch_list'], final_training_results['val_kappa_list'], 'r--', label='Kappa')
plt.title('F1 Macro & Kappa')
plt.xlabel('Epoch')
plt.ylabel('Score')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_PATH, 'final_best_model_training_curves.png'))
plt.show()

In [ ]:
# 在训练集上进行全面评估
print("在训练集上进行全面评估...")
model.eval()
train_preds = []
train_targets = []
train_probs_all = []  # 所有类别的概率

# 使用train_loader直接评估
with torch.no_grad():
    for data, target in tqdm(train_loader, desc="训练集评估"):
        data, target = data.to(device), target.to(device)
        output = model(data)
        probs = torch.softmax(output, dim=1)
        _, preds = torch.max(output, 1)
        
        # 只评估非背景像素
        valid_mask = target != -1
        train_preds.extend(preds[valid_mask].cpu().numpy())
        train_targets.extend(target[valid_mask].cpu().numpy())
        train_probs_all.extend(probs[valid_mask].cpu().numpy())  # 保存所有类别的概率

# 计算各种评估指标
train_accuracy = accuracy_score(train_targets, train_preds)
train_balanced_acc = balanced_accuracy_score(train_targets, train_preds)
train_f1_macro = f1_score(train_targets, train_preds, average='macro')
train_f1_weighted = f1_score(train_targets, train_preds, average='weighted')
train_kappa = cohen_kappa_score(train_targets, train_preds)

train_report = classification_report(train_targets, train_preds)
train_conf_matrix = confusion_matrix(train_targets, train_preds)

# 打印主要评估指标
print(f"训练集准确率: {train_accuracy:.4f}")
print(f"训练集平衡准确率: {train_balanced_acc:.4f}")
print(f"训练集宏平均F1: {train_f1_macro:.4f}")
print(f"训练集加权F1: {train_f1_weighted:.4f}")
print(f"训练集Kappa系数: {train_kappa:.4f}")

print("\n分类报告:")
print(train_report)
print("\n混淆矩阵:")
print(train_conf_matrix)

# 可视化类别分布
plt.figure(figsize=(10, 6))
plt.subplot(1, 2, 1)
# 统计每个类别的样本数
class_counts = np.bincount(train_targets)
classes = np.arange(len(class_counts))
plt.bar(classes, class_counts)
plt.xlabel('Class')
plt.ylabel('Sample Count')
plt.title('Class Distribution in Training Set')
plt.grid(True)

# 统计每个类别的预测准确率
plt.subplot(1, 2, 2)
class_accuracy = []
for cls in range(len(class_counts)):
    if cls in np.unique(train_targets):
        mask = train_targets == cls
        if np.sum(mask) > 0:
            acc = np.mean(np.array(train_preds)[mask] == cls)
            class_accuracy.append(acc)
        else:
            class_accuracy.append(0)
    else:
        class_accuracy.append(0)

plt.bar(np.arange(len(class_accuracy)), class_accuracy)
plt.xlabel('Class')
plt.ylabel('Accuracy')
plt.title('Per-Class Accuracy')
plt.grid(True)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_PATH, 'training_set_class_distribution.png'))
plt.show()

# 创建多分类混淆矩阵可视化
plt.figure(figsize=(12, 10))
# 使用对数缩放更好地显示不平衡的混淆矩阵
conf_mat_log = np.log1p(train_conf_matrix)  # log(1+x)以处理零值
mask = train_conf_matrix == 0
sns.heatmap(conf_mat_log, annot=False, fmt='d', cmap='Blues',
            mask=mask)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix (log scale)')
plt.savefig(os.path.join(SAVE_PATH, 'training_confusion_matrix_multiclass.png'))
plt.show()

# 计算每个类别的精确率、召回率和F1分数
class_precision = precision_score(train_targets, train_preds, average=None, zero_division=0)
class_recall = recall_score(train_targets, train_preds, average=None, zero_division=0)
class_f1 = f1_score(train_targets, train_preds, average=None, zero_division=0)

# 可视化类别性能
plt.figure(figsize=(15, 6))
# 只选择有实际样本的类别
valid_classes = np.where(class_counts > 0)[0]
x = np.arange(len(valid_classes))
width = 0.25

plt.bar(x - width, class_precision[valid_classes], width, label='Precision')
plt.bar(x, class_recall[valid_classes], width, label='Recall')
plt.bar(x + width, class_f1[valid_classes], width, label='F1')

plt.xlabel('Class')
plt.ylabel('Score')
plt.title('Per-Class Performance Metrics')
plt.xticks(x, valid_classes)
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_PATH, 'training_set_class_performance.png'))
plt.show()

# 保存训练集评估报告
train_report_complete = f"""
# 训练集多分类全面评估报告

## 性能指标
- 准确率: {train_accuracy:.4f}
- 平衡准确率: {train_balanced_acc:.4f}
- 宏平均F1: {train_f1_macro:.4f}
- 加权F1: {train_f1_weighted:.4f}
- Kappa系数: {train_kappa:.4f}

## 样本分布
- 总样本数: {len(train_targets)}
- 类别数量: {len(np.unique(train_targets))}
- 样本最多的类别: {np.argmax(class_counts)} ({np.max(class_counts)} 个样本)
- 样本最少的类别: {np.argmin(class_counts[class_counts > 0])} ({np.min(class_counts[class_counts > 0])} 个样本)

## 类别性能分析
"""

# 添加类别性能表格
train_report_complete += """
| 类别 | 样本数 | 精确率 | 召回率 | F1分数 |
|------|--------|--------|--------|--------|
"""

for cls in valid_classes:
    train_report_complete += f"| {cls} | {class_counts[cls]} | {class_precision[cls]:.4f} | {class_recall[cls]:.4f} | {class_f1[cls]:.4f} |\n"

train_report_complete += f"""
## 分类报告
{train_report}

## 性能分析
- 表现最好的类别: {valid_classes[np.argmax(class_f1[valid_classes])]} (F1分数: {np.max(class_f1[valid_classes]):.4f})
- 表现最差的类别: {valid_classes[np.argmin(class_f1[valid_classes])]} (F1分数: {np.min(class_f1[valid_classes]):.4f})
- 类别间性能差异: {np.max(class_f1[valid_classes]) - np.min(class_f1[valid_classes]):.4f}
"""

with open(os.path.join(SAVE_PATH, 'training_set_complete_report_multiclass.txt'), 'w') as f:
    f.write(train_report_complete)

print(f"训练集全面评估报告已保存至: {os.path.join(SAVE_PATH, 'training_set_complete_report_multiclass.txt')}")